# Fine-tune Laya on Indian bank narrations (track-my-money)

Trains `convaiinnovations/laya` to put Indian bank-statement narrations into our 32 categories.
The training data is **synthetic** (no real transactions) and is made by `backend/training/generate_dataset.py`.

**Before you run:** in the right panel set **Accelerator → GPU T4 x2**, turn **Internet on**,
and add your dataset with `train.jsonl` and `eval.jsonl` (**Add Input**).
Then **Run All**. Training takes roughly 30–90 minutes.

## 1. Settings
For the first round keep `BASE` as the published model. For a later round that continues from your previous checkpoint, set `BASE` to that checkpoint's folder (added as an input) and use the lower learning rates.

In [ ]:
BASE = "convaiinnovations/laya"   # or e.g. "/kaggle/input/laya-finance-r1/laya-finance" to continue training
OUTPUT = "/kaggle/working/laya-finance"
EPOCHS = 4
ENCODER_LR = 2.5e-5               # continuing from your own checkpoint? use 1e-5
HEAD_LR = 1e-4                    # continuing from your own checkpoint? use 5e-5
SEED = 0

## 2. Install Laya (pinned) and check the GPU

In [ ]:
!pip install -q "laya==0.4.11"
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2)"

## 3. Find the training data

In [ ]:
import glob, json, collections
train = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
evals = sorted(glob.glob("/kaggle/input/**/eval.jsonl", recursive=True))
assert train and evals, "Add the dataset with train.jsonl and eval.jsonl as an input"
TRAIN, EVAL = train[0], evals[0]
rows = [json.loads(line) for line in open(TRAIN, encoding="utf-8")]
print(TRAIN, len(rows), "rows")
print(collections.Counter(r["expected"]["category"] for r in rows).most_common(5), "...")
print("Example:", rows[0]["state"]["narration"], "->", rows[0]["expected"]["category"])

## 4. Check the data (no training yet)
`--dry-run` parses every row exactly as training will and reports anything skipped.

In [ ]:
!laya-train --data "$TRAIN" --eval "$EVAL" --base "$BASE" --dry-run

## 5. Train
Laya scores the evaluation file **before and after** training, so the output shows how much fine-tuning helped.

In [ ]:
!laya-train --data "$TRAIN" --eval "$EVAL" --base "$BASE" --out "$OUTPUT"     --epochs $EPOCHS --encoder-lr $ENCODER_LR --head-lr $HEAD_LR --seed $SEED --device cuda

## 6. Package the checkpoint for download
Download `laya-finance.zip` from the **Output** panel and unzip it into `models/laya-finance/` in the project.

In [ ]:
import os, shutil
print(sorted(os.listdir(OUTPUT)))
shutil.make_archive("/kaggle/working/laya-finance", "zip", OUTPUT)
print("Size:", round(os.path.getsize("/kaggle/working/laya-finance.zip") / 1e9, 2), "GB")